# Smart Health Testing — Hannum (full automatic pipeline)

End-to-end automatic inference on the **Hannum Smart Health** held-out test cases
(volunteers 46, 47, 49, 50 — the cases in Dataset100 `imagesTs`), then hand off to
`eval_util.process_folders` for the same MD / FA / Dice / Hausdorff analysis used in the
inference and inter-reader notebooks.

**Pipeline per case** (starts from the FULL native-spacing image):
1. **YOLO crop** on the full avg image → whole-heart square box.
2. **Crop + resize to 256** + per-channel normalization → `_0000.._0003`.
3. **nnUNet LV** (Dataset100, ResEnc-M, DA5-100ep) — **5-fold ensemble** → LV mask.
4. **YOLO RVIP (multi-contrast)** on the 256 crop → 2 insertion points (labels 2/3).
5. **Combine** LV(=1) + IPs(=2/3).
6. **`process_folders`** → un-crop back to original spacing, pull original MD/FA, compare vs GT.

GT is built from the copied test data (`06_Segmentation_Masks_CI`), 256 crop space,
LV=1 / IP1=2 / IP2=3 — named identically to the predictions.

In [ ]:
import os
from datetime import datetime

# ============================ nnUNet LV model (Dataset100) ============================
dataset_id_lv = 100
trainer   = 'nnUNetTrainerDA5_100epochs'
plans     = 'nnUNetResEncUNetMPlans'          # ResEnc-M
lv_config = '2d'
# 5-fold ENSEMBLE: pass all folds (nnUNet ensembles them). Requires folds 0..4 trained.
folds     = '0 1 2 3 4'

# ============================ YOLO weights ============================
YOLO_CROP_WEIGHTS = '/home/sastocke/nnUNet/yolo_datasets/runs/crop/weights/best.pt'
YOLO_RVIP_WEIGHTS = '/home/sastocke/nnUNet/yolo_datasets/runs/rvip_mc/weights/best.pt'  # MULTI-CONTRAST
CROP_CHANNELS     = (0, 0, 0)      # crop detector: first contrast only
RVIP_CHANNELS     = (0, 1, 2)      # RVIP multi-contrast: avg / MD / eigenvector
RVIP_SINGLE_CLASS = True
RVIP_FLIP_ASSIGN  = False          # anterior = higher; flip if a GT check shows it reversed

# ============================ data ============================
# imagesTs supplies the case list (volunteers 46,47,49,50). The pipeline itself runs on
# the FULL images in the copied test data.
IMAGES_TS      = f'/home/sastocke/nnUNet/nnUNet_raw/Dataset{dataset_id_lv}_HannumSmarthHealthDataLV/imagesTs'
TEST_DATA_ROOT = '/home/sastocke/nnUNet/HannumSmartHealthTest'          # same layout as prep source
main_folder    = os.path.dirname(TEST_DATA_ROOT)                        # process_folders builds main_folder/<prefix>/Volunteer_XX/...
annotator      = os.path.basename(TEST_DATA_ROOT)                       # prefix == folder name (single token, no underscores)

# process_folders splits the case name on '_' and uses parts[0] as the prefix -> must be one token.
assert '_' not in annotator, f"annotator '{annotator}' has underscores; rename the test folder to one token."

# ============================ outputs ============================
run       = 'SmartHealthTestHannum'
base_out  = f'/home/sastocke/nnUNet/inferenceTest_{run}'
output_crop_masks   = f'{base_out}/CropInference'          # YOLO crop boxes as masks (== mask_inference_folder)
output_image_folder = f'{base_out}/processed256/imagesTr'  # cropped+resized _0000.._0003 (nnUNet LV input)
output_LV           = f'{base_out}/OutputLV'
output_IPs          = f'{base_out}/OutputIPs'
output_final        = f'{base_out}/Output_Final_Combined'  # pred_folder for process_folders
gt_folder           = f'{base_out}/GT_Combined'            # 256 GT (LV=1, IP=2/3), named like predictions
inspection_folder   = f'{base_out}/inspection'
log_file_path       = f'{base_out}/{run}.log'

for d in [output_crop_masks, output_image_folder, output_LV, output_IPs,
          output_final, gt_folder, inspection_folder]:
    os.makedirs(d, exist_ok=True)

def clear_dirs(dirs):
    for d in dirs:
        if os.path.isdir(d):
            for f in os.listdir(d):
                p = os.path.join(d, f)
                if os.path.isfile(p) or os.path.islink(p):
                    os.unlink(p)
            print('cleared', d)
clear_dirs([output_crop_masks, output_image_folder, output_LV, output_IPs, output_final, gt_folder])


In [ ]:
import glob, re
import numpy as np
import nibabel as nib
import cv2
import matplotlib.pyplot as plt

# repo modules (this notebook lives next to them, on the crop-alignment-and-hap-fixes branch)
import yolo_pipeline as yp
from eval_util import process_folders, dice_score_original

# nnUNet env
os.environ["nnUNet_raw"]          = "/home/sastocke/nnUNet/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/home/sastocke/nnUNet/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/home/sastocke/nnUNet/nnUNet_results"

interpolation_size = 256

# --- per-channel normalization: MUST match the prep script the LV model was trained on ---
def normalize_avg_data(x):
    return (x - np.min(x)) / (np.max(x) - np.min(x))
def normalize_mean_diff_data(x):
    return (x - 0) / 4.0                      # MD physical ceiling
def normalize_eigenvector_data(x):
    return x / np.sqrt(2)                      # X+Y capped at sqrt(2) (unit eigenvector)

def make_square_crop(mask, target_size=256):
    """Square bounding box around the non-zero pixels of `mask` (same as inference.ipynb)."""
    coords = np.argwhere(mask > 0)
    x_min, y_min = coords.min(axis=0)
    x_max, y_max = coords.max(axis=0) + 1
    side = max(x_max - x_min, y_max - y_min)
    cx, cy = (x_min + x_max) // 2, (y_min + y_max) // 2
    x_min = max(0, cx - side // 2); x_max = x_min + side
    y_min = max(0, cy - side // 2); y_max = y_min + side
    return x_min, x_max, y_min, y_max, side / target_size

def parse_case(fname):
    """From an imagesTs / mask filename -> (volunteer, divo, slice3). Prefix-agnostic."""
    stem = re.sub(r'(_0000)?\.nii(\.gz)?$', '', fname)
    parts = stem.split('_')
    i = parts.index('Volunteer')
    volunteer = f'{parts[i]}_{parts[i+1]}'                 # Volunteer_XX
    divo      = f'{parts[i+2]}_{parts[i+3]}_{parts[i+4]}'  # DiVO_dd_dd  (or MDDW_dd_dd)
    j = parts.index('slice')
    slc = f'{int(parts[j+1]):03d}'
    return volunteer, divo, slc

def case_id(volunteer, divo, slc):
    return f'{annotator}_{volunteer}_{divo}_slice_{slc}'

def source_dir(volunteer, divo):
    return os.path.join(TEST_DATA_ROOT, volunteer, 'Distortion_Corrected', divo)

# ---- discover the test cases from imagesTs (volunteers 46,47,49,50) ----
cases = []
for f in sorted(glob.glob(os.path.join(IMAGES_TS, '*_0000.nii.gz'))):
    cases.append(parse_case(os.path.basename(f)))
print(f'{len(cases)} test cases from imagesTs')
for c in cases[:8]:
    print('  ', c)


## 1. Build combined GT (LV=1, IP1=2, IP2=3) from `06_Segmentation_Masks_CI`

256 crop space, named identically to the predictions, so `process_folders` matches them
1:1. Insertion points stay as labelled regions; the point comparison is done by
`process_folders` as YOLO-point-center vs GT-IP centroid (`center_of_mass` of labels 2/3).

In [ ]:
def combine_gt_mask(mask3):
    """(256,256,3) [LV, IP1, IP2]  ->  single mask LV=1, IP1=2, IP2=3 (as in the prep)."""
    out = np.zeros(mask3.shape[:2], dtype=np.uint8)
    out[mask3[:, :, 0] > 0] = 1
    out[mask3[:, :, 1] > 0] = 2
    out[mask3[:, :, 2] > 0] = 3
    return out

built = 0
for volunteer, divo, slc in cases:
    gt_src = os.path.join(source_dir(volunteer, divo),
                          '06_Segmentation_Masks_CI', f'Cropped_Segmentation_Slice_{slc}.nii')
    if not os.path.exists(gt_src):
        print('missing GT mask:', gt_src); continue
    m = nib.load(gt_src)
    combined = combine_gt_mask(m.get_fdata())
    out = os.path.join(gt_folder, case_id(volunteer, divo, slc) + '.nii.gz')
    nib.save(nib.Nifti1Image(combined, m.affine), out)
    built += 1
print(f'built {built} combined GT masks -> {gt_folder}')


## 2. YOLO crop  →  crop + resize to 256 + normalize  (Stages 1–2)

Runs the YOLO crop detector on the FULL avg image, saves the predicted square box as a mask
(so `process_folders` can un-crop the prediction with it), then crops/​resizes/​normalizes the
four modalities to 256 and writes `_0000.._0003` — the nnUNet LV input.

In [ ]:
crop_model = yp.load_model(YOLO_CROP_WEIGHTS)
metrics_data = []           # kept for parity with inference.ipynb (crop Dice etc.)

for volunteer, divo, slc in cases:
    sd  = source_dir(volunteer, divo)
    img_folder = os.path.join(sd, '03_Segmentation_Images')      # FULL, native spacing
    avg_f = os.path.join(img_folder, f'Average_Diffusion_Weighted_Image_Slice_{slc}.nii')
    md_f  = os.path.join(img_folder, f'Mean_Diffusivty_Image_Slice_{slc}.nii')
    eig_f = os.path.join(img_folder, f'Primary_Eigenvector_Image_Slice_{slc}.nii')
    fa_f  = os.path.join(img_folder, f'Fractional_Anisotropy_Image_Slice_{slc}.nii')
    if not all(os.path.exists(p) for p in [avg_f, md_f, eig_f, fa_f]):
        print('missing modality for', volunteer, divo, slc); continue

    avg_img = nib.load(avg_f); avg = avg_img.get_fdata()
    md_ = nib.load(md_f).get_fdata()
    eig = nib.load(eig_f).get_fdata()
    fa  = nib.load(fa_f).get_fdata()

    # --- YOLO crop on the full avg image -> square box -> save as a crop mask ---
    box = yp.detect_crop_box(crop_model, avg)                 # (x_min,x_max,y_min,y_max,scale)
    if box is None:
        print('no crop detected -> skipping', volunteer, divo, slc); continue
    crop_mask = yp.crop_box_to_mask(box, avg.shape)
    cid = case_id(volunteer, divo, slc)
    nib.save(nib.Nifti1Image(crop_mask.astype(np.uint8), avg_img.affine),
             os.path.join(output_crop_masks, cid + '.nii.gz'))

    # --- crop the four modalities to the square box, resize to 256, normalize ---
    x0, x1, y0, y1, _ = make_square_crop(crop_mask)
    avg_c = cv2.resize(avg[x0:x1, y0:y1], (interpolation_size,)*2, interpolation=cv2.INTER_LINEAR)
    md_c  = cv2.resize(md_[x0:x1, y0:y1], (interpolation_size,)*2, interpolation=cv2.INTER_LINEAR)
    eig_c = cv2.resize(eig[x0:x1, y0:y1, 0] + eig[x0:x1, y0:y1, 1],
                       (interpolation_size,)*2, interpolation=cv2.INTER_LINEAR)
    fa_c  = cv2.resize(fa[x0:x1, y0:y1], (interpolation_size,)*2, interpolation=cv2.INTER_LINEAR)

    avg_c = normalize_avg_data(avg_c)
    md_c  = normalize_mean_diff_data(md_c)
    eig_c = normalize_eigenvector_data(eig_c)
    # FA already [0,1] -> leave as-is

    aff = np.eye(4)
    nib.save(nib.Nifti1Image(avg_c, aff), os.path.join(output_image_folder, f'{cid}_0000.nii.gz'))
    nib.save(nib.Nifti1Image(md_c,  aff), os.path.join(output_image_folder, f'{cid}_0001.nii.gz'))
    nib.save(nib.Nifti1Image(eig_c, aff), os.path.join(output_image_folder, f'{cid}_0002.nii.gz'))
    nib.save(nib.Nifti1Image(fa_c,  aff), os.path.join(output_image_folder, f'{cid}_0003.nii.gz'))
    print('cropped+resized', cid)
print('Stage 1-2 done ->', output_image_folder)


## 3. nnUNet LV segmentation — 5-fold ensemble (Stage 3)

Ensemble over folds 0–4 (no `-f` → nnUNet averages them). Requires all five folds trained.

In [ ]:
cmd = (f"nnUNetv2_predict -i {output_image_folder} -o {output_LV} "
       f"-d {dataset_id_lv} -c {lv_config} -tr {trainer} -p {plans} -f {folds}")
print(cmd)
with open(log_file_path, 'a') as lf:
    lf.write(f"[{datetime.now()}] LV ensemble: {cmd}\n")
os.system(cmd)


## 4. YOLO RVIP (multi-contrast) → insertion-point mask (Stage 4)

Runs on the 256 crop (`_0000` resolves the sibling `_0001/_0002` for channels `(0,1,2)`).
Anterior → label 2, inferior → label 3, to match the GT convention (IP1=2, IP2=3).

In [ ]:
rvip_model = yp.load_model(YOLO_RVIP_WEIGHTS)

for volunteer, divo, slc in cases:
    cid = case_id(volunteer, divo, slc)
    img0 = os.path.join(output_image_folder, f'{cid}_0000.nii.gz')
    if not os.path.exists(img0):
        continue
    pts = yp.detect_rvips_from_path(rvip_model, img0, channels=RVIP_CHANNELS,
                                    single_class=RVIP_SINGLE_CLASS, flip_assignment=RVIP_FLIP_ASSIGN)
    ip_mask = yp.rvips_to_mask(pts, (interpolation_size, interpolation_size),
                               anterior_label=2, inferior_label=3)
    nib.save(nib.Nifti1Image(ip_mask.astype(np.uint8), np.eye(4)),
             os.path.join(output_IPs, cid + '.nii.gz'))
    print('RVIP', cid, pts)
print('Stage 4 done ->', output_IPs)


## 5. Combine LV + IPs (Stage 5)

LV stays 1; insertion points become 2 / 3 — same convention as the GT and as `inference.ipynb`.

In [ ]:
for f in os.listdir(output_LV):
    if not f.endswith('.nii.gz'):
        continue
    lv_img = nib.load(os.path.join(output_LV, f))
    lv = lv_img.get_fdata().astype(np.int32)
    combined = lv.copy(); combined[lv == 1] = 1
    ip_path = os.path.join(output_IPs, f)
    if os.path.exists(ip_path):
        ip = nib.load(ip_path).get_fdata().astype(np.int32)
        combined[ip == 2] = 2
        combined[ip == 3] = 3
    nib.save(nib.Nifti1Image(combined, affine=lv_img.affine, header=lv_img.header),
             os.path.join(output_final, f))
print('Stage 5 done ->', output_final)


## 6. Analysis — `process_folders` (Stage 6)

Un-crops predictions (with the YOLO crop box) and GT (with the GT `02_Crop_Masks` box) back to
original spacing, pulls the original MD/FA from `03_Segmentation_Images`, and computes
Dice / Hausdorff (IPs) / median MD & FA / F1 for every case.

`interreaderName=None` → inference mode (prediction vs GT, matched by identical filename).

In [ ]:
metrics_entry_median = []
process_folders(
    pred_folder          = output_final,
    gt_folder            = gt_folder,
    main_folder          = main_folder,
    metrics_data         = metrics_data,
    metrics_entry_median = metrics_entry_median,
    interreaderName      = None,                 # inference mode
    annotator            = annotator,            # crop-mask filename prefix
    mask_inference_folder= output_crop_masks,    # YOLO crop boxes -> un-crop the predictions
)


## 7. Aggregate + save (parity with inference / inter-reader notebooks)

In [ ]:
import pandas as pd
df = pd.DataFrame(metrics_data)
df.to_excel(f'{base_out}/{run}.xlsx', index=False)
print('Mean:\n',  df.mean(numeric_only=True))
print('\nMin:\n', df.min(numeric_only=True))
print('\nMax:\n', df.max(numeric_only=True))
print(f'\nsaved {base_out}/{run}.xlsx  ({len(df)} rows)')
